# 01 - Landmark prep (run ONCE, before the per-cutoff job)
Builds, from the full cohort, everything the per-cutoff workers share:
1. the fixed 8-1-1 person split (seed 42, same `build_person_split` logic as the fusion notebook) + labels, age, episode start/end, first APO date -> `landmark_cohort_table`
2. the eligibility table (still pregnant AND no APO yet, per cutoff week) -> `eligibility_table`, so the rule is defined in ONE place
3. empty, `model_id`-partitioned metrics / sizes Delta tables (partitioning lets concurrent workers write without conflicts)
4. a one-time cache of OMOP code sets for every test-split patient (cutoff-independent, so workers don't re-query Spark)

In [0]:
!pip install torch

In [0]:
import os, pickle, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CFG = {
    "cohort_table": "catalog.schema.step7_filtered_cohort_with_apos",
    "landmark_cohort_table": "catalog.schema.protopregnet_landmark_cohort",
    "eligibility_table": "catalog.schema.protopregnet_landmark_eligibility",
    "metrics_table": "catalog.schema.protopregnet_landmark_metrics",
    "sizes_table": "catalog.schema.protopregnet_landmark_sizes",

    "export_dir": "/path/to/exports",
    "original_export_path": "/path/to/exports/full_analysis_export.pkl",  # for the split check; None to skip
    "test_code_set_cache": "/path/to/exports/landmark_test_code_sets.pkl",

    # split: identical settings to the fusion notebook
    "random_seed": 42,
    "test_size": 0.10,
    "val_size": 0.10,

    # episode end = earlier of these two (missing values ignored)
    "end_cols": ("recorded_episode_end", "inferred_episode_end"),

    # landmarks: weeks since episode_start, inclusive on both ends
    "cutoff_week_start": 0,
    "cutoff_week_end": 37,

    # eligibility rules
    "exclude_apo_on_cutoff_day": True,   # True: eligible iff dx_date > cutoff; False: iff dx_date >= cutoff
    "missing_end_is_ongoing": True,      # both end dates missing -> treated as still pregnant
}

CLASS_NAMES = ["control", "pe_ghtn", "gdm"]

In [0]:
def build_landmark_cohort(cfg):
    """Same split logic as the fusion notebook's build_person_split (person-level
    80/10/10 stratified on the (control, pe_ghtn, gdm) combo, seed 42), on the FULL
    cohort. Only extra columns (episode end, first APO date) are carried along;
    they do not touch the split calls, so membership is unchanged."""
    rec_end, inf_end = cfg["end_cols"]
    cols = ["person_id", "date_of_birth", "recorded_episode_start", "inferred_episode_start",
            rec_end, inf_end, "earliest_apo_dx_date",
            "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"]
    have = set(spark.table(cfg["cohort_table"]).columns)
    missing = [c for c in cols if c not in have]
    if missing:
        raise KeyError(f"cohort table is missing columns {missing} -- adjust CFG['end_cols'] or the table")

    # NOTE: no orderBy on purpose -- the original split was built from the unsorted pull,
    # and train_test_split depends on row order. The check cell below verifies equivalence.
    flags_df = spark.table(cfg["cohort_table"]).select(*cols).toPandas()
    flags_df["person_id"] = flags_df["person_id"].astype(str)

    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = ((flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1)
                              | (flags_df["has_ghtn"] == 1)).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)

    for c in ["date_of_birth", "recorded_episode_start", "inferred_episode_start",
              rec_end, inf_end, "earliest_apo_dx_date"]:
        flags_df[c] = pd.to_datetime(flags_df[c])
    flags_df["episode_start"] = flags_df[["recorded_episode_start", "inferred_episode_start"]].min(axis=1, skipna=True)
    flags_df["episode_end"] = flags_df[[rec_end, inf_end]].min(axis=1, skipna=True)
    flags_df["age"] = (flags_df["episode_start"] - flags_df["date_of_birth"]).dt.days / 365.25

    combo = (flags_df["y_control"] * 4 + flags_df["y_pe_ghtn"] * 2 + flags_df["y_gdm"]).astype(int)
    pid_tv, pid_test = train_test_split(
        flags_df["person_id"].values, test_size=cfg["test_size"], stratify=combo, random_state=cfg["random_seed"])
    combo_tv = combo[flags_df["person_id"].isin(pid_tv)].values
    pid_train, pid_val = train_test_split(
        pid_tv, test_size=cfg["val_size"] / (1 - cfg["test_size"]), stratify=combo_tv,
        random_state=cfg["random_seed"])

    flags_df["split"] = np.select(
        [flags_df["person_id"].isin(pid_test), flags_df["person_id"].isin(pid_val)], ["test", "val"], "train")
    assert (flags_df["split"] == "train").sum() == len(pid_train)
    flags_df["pid_int"] = flags_df["person_id"].astype("int64")
    return flags_df


t0 = time.time()
cohort = build_landmark_cohort(CFG)
print(f"{len(cohort)} people in {time.time()-t0:.0f}s; split sizes: {cohort['split'].value_counts().to_dict()}")
print(f"people with NaN age: {int(cohort['age'].isna().sum())}, no episode_start: {int(cohort['episode_start'].isna().sum())}, "
      f"no episode_end: {int(cohort['episode_end'].isna().sum())}")

In [0]:
# ---- verify the split reproduces the ORIGINAL held-out set ----
# Spark row order is not guaranteed, so we check instead of assuming: every patient in the original
# export's test set must be in the new test split, and none in train/val.
if CFG["original_export_path"] and os.path.exists(CFG["original_export_path"]):
    with open(CFG["original_export_path"], "rb") as f:
        orig_test = set(map(str, pickle.load(f)["pids_test"]))
    new_test = set(cohort.loc[cohort["split"] == "test", "person_id"])
    n_out = len(orig_test - new_test)
    if n_out > 0:
        raise RuntimeError(f"{n_out}/{len(orig_test)} original test patients are NOT in the new test split -- the split "
                           f"differs from the original (Spark row order?). Do not proceed.")
    print(f"PASSED: all {len(orig_test)} original test patients are in the new test split.")
else:
    print("WARNING: original export not found -- split equivalence NOT verified.")

In [0]:
# ---- landmark cohort table (one row per person) ----
out = cohort[["person_id", "pid_int", "split", "y_control", "y_pe_ghtn", "y_gdm", "age",
              "episode_start", "episode_end", "earliest_apo_dx_date"]].copy()
(spark.createDataFrame(out).write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true").saveAsTable(CFG["landmark_cohort_table"]))
print(f"wrote {len(out)} rows to {CFG['landmark_cohort_table']}")

In [0]:
# ---- eligibility table: (pid_int, cutoff_week) for every patient eligible at that landmark ----
def build_eligibility(cohort, cfg):
    """Eligible at cutoff k  <=>  has an episode_start AND still pregnant AND no APO yet.
      cutoff        = episode_start + 7k days
      no APO yet    : no APO date, or dx_date > cutoff (dx_date >= cutoff if exclude_apo_on_cutoff_day is False)
      still pregnant: episode_end > cutoff (missing end counts as ongoing if missing_end_is_ongoing)
    Vectorized over patients; looped over the 38 cutoffs only."""
    dx, end, start = cohort["earliest_apo_dx_date"], cohort["episode_end"], cohort["episode_start"]
    pids = cohort["pid_int"].to_numpy()
    parts = []
    for k in range(cfg["cutoff_week_start"], cfg["cutoff_week_end"] + 1):
        cutoff = start + pd.Timedelta(days=7 * k)
        no_apo_yet = dx.isna() | ((dx > cutoff) if cfg["exclude_apo_on_cutoff_day"] else (dx >= cutoff))
        ongoing = (end > cutoff) | (end.isna() if cfg["missing_end_is_ongoing"] else False)
        keep = (no_apo_yet & ongoing & start.notna()).to_numpy()
        parts.append(pd.DataFrame({"pid_int": pids[keep], "cutoff_week": np.int32(k)}))
    return pd.concat(parts, ignore_index=True)

elig = build_eligibility(cohort, CFG)
spark.createDataFrame(elig).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable(CFG["eligibility_table"])

# funnel: eligible patients and positives per cutoff (counts only)
m = elig.merge(cohort[["pid_int", "split", "y_control", "y_pe_ghtn", "y_gdm"]], on="pid_int")
funnel = m.groupby("cutoff_week").agg(eligible=("pid_int", "size"), control=("y_control", "sum"),
                                       pe_ghtn=("y_pe_ghtn", "sum"), gdm=("y_gdm", "sum"))
test_funnel = m[m["split"] == "test"].groupby("cutoff_week").agg(
    test_n=("pid_int", "size"), test_pe_ghtn=("y_pe_ghtn", "sum"), test_gdm=("y_gdm", "sum"))
print(funnel.join(test_funnel).to_string())

In [0]:
# ---- result tables (created empty; partitioned by model_id so concurrent workers don't conflict) ----
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CFG['metrics_table']} (
  model_id STRING, cutoff_week INT, scope STRING, class_name STRING, metric STRING,
  estimate DOUBLE, ci_lo DOUBLE, ci_hi DOUBLE, n_test INT, run_ts TIMESTAMP
) USING DELTA PARTITIONED BY (model_id)""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CFG['sizes_table']} (
  model_id STRING, cutoff_week INT, split STRING, population STRING,
  n_patients INT, n_control INT, n_pe_ghtn INT, n_gdm INT, run_ts TIMESTAMP
) USING DELTA PARTITIONED BY (model_id)""")
print("metrics / sizes tables ready")

In [0]:
# ---- one-time OMOP code-set cache for ALL test-split patients (cutoff-independent) ----
OMOP_TABLES = {
    "condition": ("catalog.schema.condition_occurrence_cohort", "condition_concept_id", "condition_source_concept_id"),
    "procedure": ("catalog.schema.procedure_occurrence_cohort", "procedure_concept_id", "procedure_source_concept_id"),
    "drug": ("catalog.schema.drug_exposure_cohort", "drug_concept_id", "drug_concept_id"),
    "measurement": ("catalog.schema.measurement_cohort", "measurement_concept_id", "measurement_source_concept_id"),
}

def fetch_code_sets_batched(person_ids):
    codes_by_patient = {str(p): set() for p in person_ids}
    id_list_sql = ", ".join(f"'{p}'" for p in person_ids)
    for domain, (table, concept_col, source_col) in OMOP_TABLES.items():
        df = spark.sql(f"""
            SELECT person_id, {concept_col} AS concept_id, {source_col} AS source_concept_id
            FROM {table} WHERE person_id IN ({id_list_sql})
        """).toPandas()
        if df.empty:
            continue
        cid = df["concept_id"].fillna(0).astype(int)
        sid = df["source_concept_id"].fillna(0).astype(int)
        df["resolved_id"] = cid.where(cid != 0, sid)
        df = df[df["resolved_id"] != 0]
        for pid, rid in zip(df["person_id"].astype(str), df["resolved_id"]):
            if pid in codes_by_patient:
                codes_by_patient[pid].add((domain, int(rid)))
    return codes_by_patient

test_ids = cohort.loc[cohort["split"] == "test", "person_id"].tolist()
cache = fetch_code_sets_batched(test_ids)
with open(CFG["test_code_set_cache"], "wb") as f:
    pickle.dump(cache, f)
print(f"cached code sets for {len(cache)} test patients -> {CFG['test_code_set_cache']}")